# Módulo 00 — internals do modelo: a base cheia

O [walkthrough](srag_model_walkthrough.ipynb) treina na amostra commitada.
Este caderno paga a conta completa: constrói o Ouro inteiro (1,28 M) a
partir do Bronze quando preciso, treina nos 1,24 M e responde as perguntas
que só a base cheia responde — **quanto custou amostrar o treino**, como a
AUC se comporta ano a ano, e se o paciente-exemplo da amostra sobrevive ao
modelo cheio.

In [1]:
%pip install -q pandas pyarrow scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys
import tempfile
import time
import urllib.request

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "srag_model.py").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "srag_model.py").exists():
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_gold as GD
import srag_model as M
import srag_silver as S

DATA = pathlib.Path.home() / "Documents/srag-data"
LOCAL = (DATA / "gold/gold_covid_obito.parquet").exists()
print("Ouro local" if LOCAL else "Ouro será construído do Bronze (S3), ano a ano")

Ouro local


## 1. O Ouro cheio

Local quando existe; senão, Bronze→`S.build`→`GD.build_gold` por ano em
diretório temporário — a mesma decisão congelada do manifesto, sem nenhum
default escondido.

In [3]:
DECISOES = GD.Decisions(
    coorte="hospitalizado", etiologia="covid-amplo",
    alvo="obito-casos-fechados", inicio=pd.Timestamp("2020-02-26"),
    idade="todas", idade_maxima=120.0, idade_ausente="excluir",
    nosocomial="manter", split="temporal:2022-12-31/2023/2024",
    amostra_treino=200_000, semente=42,
)

if LOCAL:
    ouro = pd.read_parquet(DATA / "gold/gold_covid_obito.parquet")
else:
    S3 = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG"
    tmp = pathlib.Path(tempfile.gettempdir()) / "srag-bronze"
    tmp.mkdir(parents=True, exist_ok=True)

    def frames():
        for ano in range(2019, 2025):
            nome = f"INFLUD{ano % 100:02d}-26-06-2025.parquet"
            alvo = tmp / nome
            if not alvo.exists():
                urllib.request.urlretrieve(f"{S3}/{ano}/{nome}", alvo)
            silver, _quar = S.build(alvo)
            yield silver[[c for c in GD.NEEDED if c in silver.columns]]
            alvo.unlink()

    ouro, _counts = GD.build_gold(list(frames()), DECISOES)
for c in M.CATEGORICAS:
    ouro[c] = ouro[c].astype("category")
print(f"Ouro: {len(ouro):,} × {len(ouro.columns)}")
print(ouro.groupby("split", observed=True)["y_obito"].agg(["size", "sum", "mean"]).round(3).to_string())

Ouro: 1,282,970 × 50
          size     sum   mean
split                        
test     16142    2945  0.182
train  1242680  390769  0.314
val      24148    4572  0.189


## 2. O preço da amostra: fit cheio vs fit de 200 mil

In [4]:
t0 = time.time()
xgb_cheio, logit_cheio = M.fit_models(ouro)
t_cheio = time.time() - t0

amostra = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
t0 = time.time()
xgb_amostra, _ = M.fit_models(amostra)
t_amostra = time.time() - t0

linhas = []
for nome, modelo, base, t in (
    ("cheio (1,24 M)", xgb_cheio, ouro, t_cheio),
    ("amostra (200 k)", xgb_amostra, amostra, t_amostra),
):
    for s in ("val", "test"):
        linhas.append({"treino": nome, "fit_s": round(t, 1)} | M.metrics(modelo, base, s))
print(pd.DataFrame(linhas).to_string(index=False))

         treino  fit_s split     n  obitos    auc  brier  previsto_medio  observado
 cheio (1,24 M)   19.3   val 24148    4572 0.7614 0.1354          0.2235     0.1893
 cheio (1,24 M)   19.3  test 16142    2945 0.7674 0.1307          0.2188     0.1824
amostra (200 k)    2.4   val 24148    4572 0.7522 0.1371          0.2231     0.1893
amostra (200 k)    2.4  test 16142    2945 0.7575 0.1326          0.2164     0.1824


O delta amostra→cheio é o número que autoriza o desenho inteiro do curso:
se ele for pequeno, explicar o modelo-da-amostra é explicar o modelo. A
avaliação é idêntica nos dois casos (val e teste inteiros, por construção
da amostra).

## 3. AUC por ano de início — a deriva, medida onde dói

In [5]:
from sklearn.metrics import roc_auc_score

x_xgb, _, y = M.design_matrix(ouro)
p = xgb_cheio.predict_proba(x_xgb)[:, 1]
por_ano = []
for ano in sorted(ouro["ano_onset"].unique()):
    m = (ouro["ano_onset"] == ano).to_numpy()
    if ouro.loc[m, "y_obito"].nunique() < 2:
        continue
    por_ano.append({
        "ano": int(ano), "n": int(m.sum()),
        "CFR %": round(100 * float(ouro.loc[m, "y_obito"].mean()), 1),
        "AUC": round(float(roc_auc_score(y[m], p[m])), 4),
        "previsto médio": round(float(p[m].mean()), 3),
    })
print(pd.DataFrame(por_ano).to_string(index=False))

 ano      n  CFR %    AUC  previsto médio
2020 407679   32.2 0.7926           0.322
2021 723037   31.4 0.7582           0.314
2022 111964   28.8 0.7520           0.289
2023  24148   18.9 0.7614           0.223
2024  16142   18.2 0.7674           0.219


A discriminação (AUC) resiste melhor que a calibração: o ranking de risco
viaja entre regimes, o nível absoluto não — exatamente a leitura que o
walkthrough §2 faz no teste.

## 4. Curva de aprendizado — quanto treino compra o quê

In [6]:
frac_alvo = [50_000, 200_000, len(ouro[ouro["split"] == "train"])]
rng = np.random.default_rng(42)
treino_cheio = ouro[ouro["split"] == "train"].reset_index(drop=True)
aval = ouro[ouro["split"] != "train"]
curva = []
for n_treino in frac_alvo:
    if n_treino < len(treino_cheio):
        idx = np.sort(rng.choice(len(treino_cheio), size=n_treino, replace=False))
        base = pd.concat([treino_cheio.iloc[idx], aval], ignore_index=True)
    else:
        base = ouro
    for c in M.CATEGORICAS:
        base[c] = base[c].astype("category")
    mdl, _ = M.fit_models(base)
    curva.append({"treino": n_treino} | {
        k: v for k, v in M.metrics(mdl, base, "test").items() if k in ("auc", "brier")
    })
print(pd.DataFrame(curva).to_string(index=False))

 treino    auc  brier
  50000 0.7446 0.1358
 200000 0.7547 0.1339
1242680 0.7674 0.1307


## 5. O exemplar sobrevive ao modelo cheio?

In [7]:
ex_amostra = M.pick_exemplar(xgb_amostra, amostra)
ex_cheio = M.pick_exemplar(xgb_cheio, ouro)
print(f"regra na amostra  → gold_id {int(ex_amostra.gold_id)}  p = {ex_amostra.p_obito:.3f}")
print(f"regra na base cheia → gold_id {int(ex_cheio.gold_id)}  p = {ex_cheio.p_obito:.3f}")
p_do_exemplar = float(M.predict_proba(
    xgb_cheio, ouro[ouro["gold_id"] == int(ex_amostra.gold_id)]
)[0])
print(f"p do exemplar-da-amostra sob o modelo cheio: {p_do_exemplar:.3f}")
print("\nO paciente do curso é o da AMOSTRA — o que todos os módulos refazem.")
print("Se as regras divergem, é registro, não problema: |p-0,5| mínimo é")
print("sensível ao modelo, e o modelo-da-amostra é o objeto de estudo.")

regra na amostra  → gold_id 1269214  p = 0.500
regra na base cheia → gold_id 1269556  p = 0.500
p do exemplar-da-amostra sob o modelo cheio: 0.416

O paciente do curso é o da AMOSTRA — o que todos os módulos refazem.
Se as regras divergem, é registro, não problema: |p-0,5| mínimo é
sensível ao modelo, e o modelo-da-amostra é o objeto de estudo.


## 6. As quatro impossibilidades: amostra vs base cheia

A regra do repositório: medir no cheio E re-medir na amostra, imprimir os
dois — uma diferença além de composição é achado sobre a amostra.

In [8]:
def fracoes(base):
    n = len(base)
    return {
        "portão do funil": 100 * int((~base["fator_risc_portao"].astype(bool)).sum()) / n,
        "pré-campanha": 100 * int((base["meses_desde_mar2020"] < M.MESES_CAMPANHA).sum()) / n,
        "critério-2 por um fio": 100 * int((base["n_crit2"] == 1).sum()) / n,
        "critério-3 por um fio": 100 * int((base["n_crit3"] == 1).sum()) / n,
    }

quadro = pd.DataFrame({
    "base cheia %": fracoes(ouro), "amostra %": fracoes(amostra),
}).round(2)
quadro["Δ"] = (quadro["amostra %"] - quadro["base cheia %"]).round(2)
print(quadro.to_string())
print("\nA amostra sub-representa o pré-campanha porque val/test (2023-24,")
print("inteiros) pesam mais nela do que na base — efeito de composição")
print("declarado; os módulos citam a fração da base em que operam.")

                       base cheia %  amostra %     Δ
portão do funil               38.05      37.27 -0.78
pré-campanha                  34.50      29.67 -4.83
critério-2 por um fio         78.83      79.60  0.77
critério-3 por um fio         23.66      24.67  1.01

A amostra sub-representa o pré-campanha porque val/test (2023-24,
inteiros) pesam mais nela do que na base — efeito de composição
declarado; os módulos citam a fração da base em que operam.
